# CFP_Data_Download

Download 24 asset return series from Yahoo Finance.

**Paper:** Pele, Lessmann, Hardle (2026)

**Dependencies:** `yfinance`, `pandas`, `numpy`

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

OUT = Path('results')
OUT.mkdir(exist_ok=True)

TICKERS = {
    'SP500': '^GSPC', 'STOXX': '^STOXX', 'GDAXI': '^GDAXI', 'CACT': '^FCHI',
    'FTSE100': '^FTSE', 'ICLN': 'ICLN', 'NIKKEI': '^N225', 'HSI': '^HSI',
    'BOVESPA': '^BVSP', 'NIFTY': '^NSEI', 'ASX200': '^AXJO',
    'CBU0': 'CBU0.L', 'TLT': 'TLT', 'IBGL': 'IBGL.L',
    'DJCI': 'DJCI', 'GOLD': 'GC=F', 'WTI': 'CL=F', 'NATGAS': 'NG=F',
    'BTC': 'BTC-USD', 'ETH': 'ETH-USD',
    'EURUSD': 'EURUSD=X', 'GBPUSD': 'GBPUSD=X', 'USDJPY': 'JPY=X', 'AUDUSD': 'AUDUSD=X',
}

START = '2000-01-01'
END = '2026-03-19'

summary = []
for label, ticker in tqdm(TICKERS.items(), desc='Downloading'):
    df = yf.download(ticker, start=START, end=END, progress=False)
    if df.empty:
        print(f'  WARNING: {label} ({ticker}) returned no data')
        continue
    df = df[['Close']].dropna().rename(columns={'Close': 'close'})
    df.index.name = 'date'
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    df = df.dropna()
    df = df[df['log_return'].abs() <= 0.50]
    df.to_csv(OUT / f'{label}.csv')
    r = df['log_return']
    summary.append({
        'asset': label, 'ticker': ticker, 'start': str(df.index.min().date()),
        'end': str(df.index.max().date()), 'n_days': len(df),
        'mean': r.mean(), 'std': r.std(), 'skew': r.skew(),
        'kurtosis': r.kurtosis(), 'min': r.min(), 'max': r.max(),
    })

df_sum = pd.DataFrame(summary)
df_sum.to_csv(OUT / 'asset_summary.csv', index=False)
print(f'\nDone: {len(summary)} assets saved to results/')
print(df_sum[['asset','n_days','mean','std','kurtosis']].round(4).to_string(index=False))